## В оригинальном наборе данных по одному ответу к вопросу

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
from typing import Dict, List, Optional, Union
import warnings

def process_single_answer_dataset(
    dataset_name: str,
    out_dir: str,
    question_columns: Union[Dict[str, str], str] = 'question',
    answer_columns: Union[Dict[str, str], str] = 'answers',
    answer_text_field: str = 'text',
    config: Optional[str] = None,
    test_size: float = 0.25,
    val_size: float = 0.5,
    dataset_tags: List[str] = None,
    clearml_project: str = "Semantic Search",
    auto_detect_splits: bool = True,
    main_split: Optional[str] = None
):
    """    
    Args:
        dataset_name: Название датасета в Hugging Face (например, 'hsseinmz/arcd')
        out_dir: Директория для сохранения обработанных данных
        question_columns: Название колонки с вопросами (str) или словарь {split: column}
        answer_columns: Название колонки с ответами (str) или словарь {split: column}
        answer_text_field: Название поля с текстом ответа (если ответы вложенные)
        config: Конфигурация датасета (если есть)
        test_size: Размер тестовой выборки (для датасетов с одним split)
        val_size: Размер валидационной выборки (для датасетов с одним split)
        dataset_tags: Теги для ClearML
        clearml_project: Название проекта в ClearML
        auto_detect_splits: Автоматически определять splits по размеру данных
        main_split: Главный split (если auto_detect_splits=False)
    """
    # Загрузка датасета
    data = load_dataset(dataset_name, config) if config else load_dataset(dataset_name)
    
    # Определение доступных splits
    splits = list(data.keys())
    print(f"Found splits: {splits}")
    
    # Нормализация параметров question_columns и answer_columns
    if isinstance(question_columns, str):
        question_columns = {split: question_columns for split in splits}
    if isinstance(answer_columns, str):
        answer_columns = {split: answer_columns for split in splits}
    
    # Автоматическое определение типа splits
    if auto_detect_splits:
        split_sizes = {split: len(data[split]) for split in splits}
        sorted_splits = sorted(split_sizes.items(), key=lambda x: x[1], reverse=True)
        
        if len(splits) == 1:
            main_split = splits[0]
        elif len(splits) >= 2:
            main_split = sorted_splits[0][0]  
            if len(splits) == 2:
                val_split = sorted_splits[1][0]
            else:
                val_split = sorted_splits[1][0]
                test_split = sorted_splits[2][0]
    
    # Если имена splits известны, используем их
    train_split = main_split if main_split else 'train'
    val_split = 'validation' if 'validation' in splits else 'val' if 'val' in splits else None
    test_split = 'test' if 'test' in splits else None
    
    # Сбор всех вопросов и ответов
    all_questions = []
    all_answers = []
    pos_examples = []
    current_id = 0
    
    # Обработка в зависимости от количества splits
    if len(splits) == 1:
        # Один split - разбиваем на train/val/test
        split = splits[0]
        questions = data[split][question_columns.get(split, 'question')]
        answers = data[split][answer_columns.get(split, 'answers')]
        
        if isinstance(answers[0], dict):  # Если ответы вложенные (как в SQUAD)
            answers = [ans[answer_text_field][0] if isinstance(ans[answer_text_field], list) else ans[answer_text_field] for ans in answers]
        
        # Создаем корпус
        corpus = Dataset.from_dict({'id': list(range(len(answers))), 'text': answers})
        
        # Создаем позитивные примеры
        pos_examples = [{'doc_id': [i]} for i in range(len(questions))]
        
        # Разбиваем на train/val/test
        ds = Dataset.from_dict({'query': questions, 'positives': pos_examples})
        ds = ds.train_test_split(test_size=val_size)
        data_val = ds['test']
        ds_test = ds['test'].train_test_split(test_size=test_size/(1-val_size))
        
        dataset = DatasetDict({
            'train': ds['train'],
            'val': ds_test['train'],
            'test': ds_test['test']
        })
        
    elif len(splits) == 2:
        # Два splits (обычно train/validation)
        train_split_name = train_split if train_split in splits else splits[0]
        val_split_name = val_split if val_split in splits else splits[1]
        
        q_train = data[train_split_name][question_columns.get(train_split_name, 'question')]
        q_val = data[val_split_name][question_columns.get(val_split_name, 'question')]
        
        a_train = data[train_split_name][answer_columns.get(train_split_name, 'answers')]
        a_val = data[val_split_name][answer_columns.get(val_split_name, 'answers')]
        
        # Обработка ответов
        def process_answers(answers):
            if isinstance(answers[0], dict):
                return [ans[answer_text_field][0] if isinstance(ans[answer_text_field], list) else ans[answer_text_field] for ans in answers]
            return answers
        
        a_train = process_answers(a_train)
        a_val = process_answers(a_val)
        
        all_answers = a_train + a_val
        corpus = Dataset.from_dict({'id': list(range(len(all_answers))), 'text': all_answers})
        
        # Позитивные примеры
        pos_train = [{'doc_id': [i]} for i in range(len(q_train))]
        pos_val = [{'doc_id': [len(q_train)+i]} for i in range(len(q_val))]
        
        data_train = Dataset.from_dict({'query': q_train, 'positives': pos_train})
        data_val = Dataset.from_dict({'query': q_val, 'positives': pos_val})
        
        # Разбиваем validation на val/test
        ds = data_val.train_test_split(test_size=0.5)
        dataset = DatasetDict({
            'train': data_train,
            'val': ds['train'],
            'test': ds['test']
        })
        
    else:
        # Три и более splits
        train_split_name = train_split if train_split in splits else splits[0]
        val_split_name = val_split if val_split in splits else splits[1]
        test_split_name = test_split if test_split in splits else splits[2]
        
        q_train = data[train_split_name][question_columns.get(train_split_name, 'question')]
        q_val = data[val_split_name][question_columns.get(val_split_name, 'question')]
        q_test = data[test_split_name][question_columns.get(test_split_name, 'question')]
        
        a_train = data[train_split_name][answer_columns.get(train_split_name, 'answers')]
        a_val = data[val_split_name][answer_columns.get(val_split_name, 'answers')]
        a_test = data[test_split_name][answer_columns.get(test_split_name, 'answers')]
        
        # Обработка ответов
        def process_answers(answers):
            if isinstance(answers[0], dict):
                return [ans[answer_text_field][0] if isinstance(ans[answer_text_field], list) else ans[answer_text_field] for ans in answers]
            return answers
        
        a_train = process_answers(a_train)
        a_val = process_answers(a_val)
        a_test = process_answers(a_test)
        
        all_answers = a_train + a_val + a_test
        corpus = Dataset.from_dict({'id': list(range(len(all_answers))), 'text': all_answers})
        
        # Позитивные примеры
        pos_train = [{'doc_id': [i]} for i in range(len(q_train))]
        pos_val = [{'doc_id': [len(q_train)+i]} for i in range(len(q_val))]
        pos_test = [{'doc_id': [len(q_train)+len(q_val)+i]} for i in range(len(q_test))]
        
        dataset = DatasetDict({
            'train': Dataset.from_dict({'query': q_train, 'positives': pos_train}),
            'val': Dataset.from_dict({'query': q_val, 'positives': pos_val}),
            'test': Dataset.from_dict({'query': q_test, 'positives': pos_test})
        })
    
    # Добавляем labels к тестовой выборке
    def add_labels(row, idx, size_col=None):
        labels = [0]*size_col
        labels[idx] = 1
        row["labels"] = list(map(bool, labels))
        return row
    
    if 'test' in dataset:
        dataset['test'] = dataset['test'].map(
            add_labels,
            with_indices=True,
            fn_kwargs={"size_col": dataset['test'].shape[0]},
        )
    else:
        warnings.warn("No test split found in the dataset")
    
    # Сохраняем результаты
    import os
    os.makedirs(out_dir, exist_ok=True)
    
    dataset.save_to_disk(f"{out_dir}/dataset")
    corpus.save_to_disk(f"{out_dir}/corpus")
    
    # Логирование в ClearML (если доступно)
    try:
        from clearml import Dataset 
        clearml_ds = Dataset.create(
            ds=dataset,
            corpus=corpus,
            ds_path=f"{out_dir}/dataset",
            finalize=False,
            dataset_name=dataset_name.split('/')[-1],
            dataset_project=clearml_project,
            dataset_tags=dataset_tags or []
        )
        clearml_ds.compute_mean_lens(dataset, title="queries", column_names=["query"], k=100000)
        clearml_ds.compute_mean_examples_count(dataset, column_names=["positives"])
    except ImportError:
        print("ClearML not available, skipping logging")
    except Exception as e:
        print(f"Error logging to ClearML: {e}")
    
    return dataset, corpus


## В оригинальном наборе данных множество правильных ответов к вопросу

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
from typing import Dict, List, Optional, Union
import os

def process_qa_dataset_with_multiple_answers(
    dataset_name: str,
    out_dir: str,
    question_column: str = 'question',
    answers_column: str = 'answers',
    answer_text_field: Optional[str] = None,
    config: Optional[str] = None,
    test_size: float = 0.25,
    val_size: float = 0.5,
    dataset_tags: List[str] = None,
    clearml_project: str = "Semantic Search"
):
    """
    
    Args:
        dataset_name: Название датасета в Hugging Face
        out_dir: Директория для сохранения результатов
        question_column: Название колонки с вопросами
        answers_column: Название колонки с ответами
        answer_text_field: Поле с текстом ответа (если ответы вложенные)
        config: Конфигурация датасета (если есть)
        test_size: Размер тестовой выборки
        val_size: Размер валидационной выборки
        dataset_tags: Теги для ClearML
        clearml_project: Название проекта в ClearML
    """
    # Загрузка датасета
    dataset = load_dataset(dataset_name, config) if config else load_dataset(dataset_name)
    
    # Определение доступных splits
    splits = list(dataset.keys())
    print(f"Available splits: {splits}")
    
    # Поддержка разных форматов splits
    if len(splits) == 1:
        train_split = splits[0]
        val_split = None
        test_split = None
    elif len(splits) == 2:
        train_split = 'train' if 'train' in splits else splits[0]
        val_split = 'validation' if 'validation' in splits else 'val' if 'val' in splits else splits[1]
        test_split = None
    else:
        train_split = 'train' if 'train' in splits else splits[0]
        val_split = 'validation' if 'validation' in splits else 'val' if 'val' in splits else splits[1]
        test_split = 'test' if 'test' in splits else splits[2]
    
    # Сбор всех вопросов и ответов
    all_questions = []
    all_answers = []
    split_boundaries = {}  # Для отслеживания границ между splits
    
    # Обработка train split
    questions_train = dataset[train_split][question_column]
    answers_train = dataset[train_split][answers_column]
    
    # Извлечение текстов ответов
    flat_answers_train = []
    for ans_group in answers_train:
        if answer_text_field:
            if isinstance(ans_group[0], dict):
                flat_answers_train.extend([ans[answer_text_field] for ans in ans_group])
            else:
                flat_answers_train.extend(ans_group)
        else:
            flat_answers_train.extend(ans_group)
    
    all_questions.extend(questions_train)
    all_answers.extend(flat_answers_train)
    split_boundaries['train'] = (0, len(questions_train))
    
    # Обработка validation split (если есть)
    if val_split:
        questions_val = dataset[val_split][question_column]
        answers_val = dataset[val_split][answers_column]
        
        flat_answers_val = []
        for ans_group in answers_val:
            if answer_text_field:
                if isinstance(ans_group[0], dict):
                    flat_answers_val.extend([ans[answer_text_field] for ans in ans_group])
                else:
                    flat_answers_val.extend(ans_group)
            else:
                flat_answers_val.extend(ans_group)
        
        all_questions.extend(questions_val)
        all_answers.extend(flat_answers_val)
        split_boundaries['val'] = (len(questions_train), len(questions_train) + len(questions_val))
    
    # Обработка test split (если есть)
    if test_split:
        questions_test = dataset[test_split][question_column]
        answers_test = dataset[test_split][answers_column]
        
        flat_answers_test = []
        for ans_group in answers_test:
            if answer_text_field:
                if isinstance(ans_group[0], dict):
                    flat_answers_test.extend([ans[answer_text_field] for ans in ans_group])
                else:
                    flat_answers_test.extend(ans_group)
            else:
                flat_answers_test.extend(ans_group)
        
        all_questions.extend(questions_test)
        all_answers.extend(flat_answers_test)
        split_boundaries['test'] = (len(questions_train) + len(questions_val if val_split else 0), 
                                  len(questions_train) + (len(questions_val if val_split else 0) + len(questions_test))
    
    # Создание корпуса
    corpus = Dataset.from_dict({
        'id': list(range(len(all_answers))),
        'text': all_answers
    })
    
    # Создание позитивных примеров
    def create_pos_examples(questions, answers, start_idx=0):
        pos_examples = []
        current_idx = start_idx
        for ans_group in answers:
            pos_l = list(range(current_idx, current_idx + len(ans_group)))
            pos_examples.append({'doc_id': pos_l})
            current_idx += len(ans_group)
        return pos_examples, current_idx
    
    # Обработка train split
    pos_train, current_idx = create_pos_examples(questions_train, answers_train)
    data_train = Dataset.from_dict({
        'query': questions_train,
        'positives': pos_train
    })
    
    # Обработка validation и test splits
    data_val, data_test = None, None
    
    if val_split:
        pos_val, current_idx = create_pos_examples(questions_val, answers_val, current_idx)
        if test_split:
            # Если есть отдельный test split
            pos_test, _ = create_pos_examples(questions_test, answers_test, current_idx)
            data_val = Dataset.from_dict({
                'query': questions_val,
                'positives': pos_val
            })
            data_test = Dataset.from_dict({
                'query': questions_test,
                'positives': pos_test
            })
        else:
            # Разбиваем validation на val/test
            ds_val = Dataset.from_dict({
                'query': questions_val,
                'positives': pos_val
            }).train_test_split(test_size=test_size)
            data_val = ds_val['train']
            data_test = ds_val['test']
    else:
        # Если только один split - разбиваем на train/val/test
        ds = data_train.train_test_split(test_size=val_size + test_size)
        ds_test = ds['test'].train_test_split(test_size=test_size/(val_size + test_size))
        data_train = ds['train']
        data_val = ds_test['train']
        data_test = ds_test['test']
    
    # Добавление labels для тестовой выборки
    def add_labels(row, idx, size_col=None):
        labels = [0]*size_col
        labels[idx] = 1
        row["labels"] = list(map(bool, labels))
        return row
    
    data_test = data_test.map(
        add_labels,
        with_indices=True,
        fn_kwargs={"size_col": data_test.shape[0]},
    )
    
    # Создание итогового DatasetDict
    final_dataset = DatasetDict({
        'train': data_train,
        'val': data_val if data_val is not None else Dataset.from_dict({'query': [], 'positives': []}),
        'test': data_test
    })
    
    # Сохранение результатов
    os.makedirs(out_dir, exist_ok=True)
    final_dataset.save_to_disk(os.path.join(out_dir, 'dataset'))
    corpus.save_to_disk(os.path.join(out_dir, 'corpus'))
    
    # Логирование в ClearML (опционально)
    try:
        from clearml import Dataset as LocalDataset
        clearml_ds = LocalDataset.create(
            ds=final_dataset,
            corpus=corpus,
            ds_path=os.path.join(out_dir, 'dataset'),
            finalize=False,
            dataset_name=dataset_name.split('/')[-1],
            dataset_project=clearml_project,
            dataset_tags=dataset_tags or []
        )
        clearml_ds.compute_mean_lens(final_dataset, title="queries", column_names=["query"], k=100000)
        clearml_ds.compute_mean_examples_count(final_dataset, column_names=["positives"])
    except ImportError:
        print("ClearML not installed, skipping logging")
    
    return final_dataset, corpus)

## Набор данных с множеством ответов, один из которых верный

In [ ]:
from datasets import Dataset, DatasetDict
from tqdm import tqdm
from typing import Dict, List, Optional
import os

def process_multi_answer_dataset(
    dataset_path: str,
    output_dir: str,
    question_field: str = 'beingAsked',
    answers_field: str = 'answerChoices',
    correct_answer_field: str = 'correctAnswer',
    answer_text_field: str = 'processedText',
    splits: Optional[Dict[str, str]] = None,
    dataset_name: Optional[str] = None,
    dataset_tags: Optional[List[str]] = None
) -> tuple:
    """
    
    Args:
        dataset_path: Путь к датасету в Hugging Face или локальный путь
        output_dir: Директория для сохранения результатов
        question_field: Название поля с вопросом
        answers_field: Название поля с вариантами ответов
        correct_answer_field: Название поля с индексом верного ответа
        answer_text_field: Название поля с текстом ответа
        splits: Словарь с именами splits {'train': ..., 'val': ..., 'test': ...}
        dataset_name: Имя датасета для сохранения
        dataset_tags: Теги датасета
    
    Returns:
        Кортеж (dataset, corpus) - обработанные данные
    """
    
    # Загрузка датасета
    if os.path.exists(dataset_path):
        dataset = DatasetDict.load_from_disk(dataset_path)
    else:
        dataset = load_dataset(dataset_path)
    
    # Определение splits
    if splits is None:
        splits = {
            'train': 'train' if 'train' in dataset else list(dataset.keys())[0],
            'val': 'validation' if 'validation' in dataset else 'val' if 'val' in dataset else list(dataset.keys())[1],
            'test': 'test' if 'test' in dataset else list(dataset.keys())[2] if len(dataset) > 2 else None
        }
    
    # Сбор всех вопросов и ответов
    all_data = {}
    corpus_answers = []
    answer_count = 0
    
    for split_name, split_key in splits.items():
        if split_key not in dataset:
            continue
            
        split_data = dataset[split_key]
        questions = []
        right_answers = []
        wrong_answers = []
        
        for item in tqdm(split_data, desc=f"Processing {split_name}"):
            # Извлечение вопроса
            question = item[question_field]
            questions.append(question)
            
            # Извлечение верного ответа
            correct_idx = item[correct_answer_field]
            answer_choices = item[answers_field]
            
            right_answer = answer_choices[correct_idx][answer_text_field]
            right_answers.append(right_answer)
            
            # Извлечение неверных ответов
            wrongs = [
                choice[answer_text_field] 
                for idx, choice in enumerate(answer_choices) 
                if idx != correct_idx
            ]
            wrong_answers.append(wrongs)
        
        all_data[split_name] = {
            'Questions': questions,
            'Right answer': right_answers,
            'Wrong answers': wrong_answers
        }
    
    # Создание корпуса всех ответов
    corpus_data = []
    for split in all_data.values():
        for right, wrongs in zip(split['Right answer'], split['Wrong answers']):
            corpus_data.append(right)
            corpus_data.extend(wrongs)
    
    corpus = Dataset.from_dict({
        'id': list(range(len(corpus_data))),
        'text': corpus_data
    })
    
    # Создание структуры для semantic search
    final_dataset = {}
    current_answer_id = 0
    
    for split_name, split_data in all_data.items():
        questions = split_data['Questions']
        positives = []
        negatives = []
        
        for right, wrongs in zip(split_data['Right answer'], split_data['Wrong answers']):
            # Позитивные примеры (верный ответ)
            positives.append({'doc_id': [current_answer_id]})
            current_answer_id += 1
            
            # Негативные примеры (неверные ответы)
            neg_ids = list(range(current_answer_id, current_answer_id + len(wrongs)))
            negatives.append({'doc_id': neg_ids})
            current_answer_id += len(wrongs)
        
        # Создание датасета для каждого split
        split_dataset = Dataset.from_dict({
            'query': questions,
            'positives': positives,
            'negatives': negatives
        })
        
        # Добавление labels для тестового набора
        if split_name == 'test':
            def add_labels(row, idx, size_col):
                labels = [0]*size_col
                labels[idx] = 1
                row["labels"] = list(map(bool, labels))
                return row
            
            split_dataset = split_dataset.map(
                add_labels,
                with_indices=True,
                fn_kwargs={"size_col": len(split_dataset)},
            )
        
        final_dataset[split_name] = split_dataset
    
    # Создание итогового DatasetDict
    final_dataset = DatasetDict(final_dataset)
    
    # Сохранение результатов
    os.makedirs(output_dir, exist_ok=True)
    final_dataset.save_to_disk(os.path.join(output_dir, 'dataset'))
    corpus.save_to_disk(os.path.join(output_dir, 'corpus'))
    
    # Логирование в ClearML (опционально)
    try:
        from clearml import Dataset as ClearMLDataset
        clearml_ds = ClearMLDataset.create(
            dataset_name=dataset_name or os.path.basename(output_dir),
            dataset_project="QA Datasets",
            dataset_tags=dataset_tags or []
        )
        clearml_ds.add_files(output_dir)
        clearml_ds.upload()
        clearml_ds.finalize()
    except ImportError:
        print("ClearML not available, skipping logging")
    
    return final_dataset, corpus


## Скачивание данных с использованием потокового чтения

In [ ]:
from huggingface_hub import hf_hub_download
from datasets import Dataset
import zstandard as zstd
import json
import io

def stream_zst_dataset(repo_id, filename):
    path = hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset")
    
    with open(path, 'rb') as f:
        dctx = zstd.ZstdDecompressor()
        stream_reader = dctx.stream_reader(f)
        
        buffer = ""
        for chunk in iter(lambda: stream_reader.read(65536), b""):
            buffer += chunk.decode('utf-8', errors='replace')
            lines = buffer.splitlines(keepends=True)
            
            for line in lines[:-1]:
                if line.strip():
                    try:
                        yield json.loads(line)
                    except json.JSONDecodeError:
                        continue
            buffer = lines[-1] if lines else ""

#Пример
dataset = Dataset.from_generator(stream_zst_dataset, gen_kwargs={
    "repo_id": "nyuuzyou/9111-questions",
    "filename": "questions.json.zst"
})


## Проверка наборов данных на предмет неправильных значений

In [ ]:
from datasets import Dataset, DatasetDict
import os
import pandas as pd
from typing import List, Dict, Set

def validate_qa_datasets(
    datasets_base_dir: str,
    dataset_names: List[str],
    save_cleaned: bool = True,
    verbose: bool = True
) -> Dict[str, Set[int]]:
    """
    Проверяет и очищает датасеты вопросов-ответов с фиксированной структурой.
    
    Args:
        datasets_base_dir: Базовая директория с датасетами
        dataset_names: Список имен датасетов для обработки
        save_cleaned: Сохранять ли очищенные версии
        verbose: Выводить прогресс
    
    Returns:
        Словарь с проблемными ID для каждого датасета
    """
    
    # Фиксированные параметры структуры
    REQUIRED_SPLITS = ['train', 'val', 'test']
    CORPUS_FILE = 'corpus'
    DATASET_FILE = 'dataset'
    TEXT_FIELD = 'text'
    QUERY_FIELD = 'query'
    ID_FIELD = 'id'
    DOC_ID_FIELD = 'doc_id'
    POSITIVES_FIELD = 'positives'
    
    bad_ids_report = {}

    for name in dataset_names:
        if verbose:
            print(f"\nProcessing dataset: {name}")
            
        dataset_dir = os.path.join(datasets_base_dir, name)
        bad_ids = set()
        
        try:
            # 1. Загрузка и проверка корпуса
            corpus_path = os.path.join(dataset_dir, CORPUS_FILE)
            corpus = Dataset.load_from_disk(corpus_path)
            
            # Проверка текстовых полей
            corpus_df = corpus.to_pandas()
            bad_text_mask = corpus_df[TEXT_FIELD].isna() | ~corpus_df[TEXT_FIELD].apply(isinstance, args=(str,))
            bad_ids.update(corpus_df[bad_text_mask][ID_FIELD].tolist())
            
            if verbose and bad_ids:
                print(f"Found {len(bad_ids)} invalid texts in corpus")

            # 2. Загрузка и проверка датасета
            dataset_path = os.path.join(dataset_dir, DATASET_FILE)
            dataset_dict = DatasetDict.load_from_disk(dataset_path)
            
            # Проверка наличия всех splits
            missing_splits = set(REQUIRED_SPLITS) - set(dataset_dict.keys())
            if missing_splits:
                print(f"Warning: Missing splits {missing_splits} in dataset {name}")
                continue
            
            cleaned_splits = {}
            
            for split in REQUIRED_SPLITS:
                split_data = dataset_dict[split].to_pandas()
                
                # Проверка query
                bad_query_mask = split_data[QUERY_FIELD].isna() | ~split_data[QUERY_FIELD].apply(isinstance, args=(str,))
                if bad_query_mask.any():
                    if verbose:
                        print(f"Found {bad_query_mask.sum()} invalid queries in {split}")
                    split_data.loc[bad_query_mask, QUERY_FIELD] = "invalid query"
                
                # Проверка ссылок на корпус
                def has_bad_refs(pos_dict):
                    if not isinstance(pos_dict, dict):
                        return True
                    return any(doc_id in bad_ids for doc_id in pos_dict.get(DOC_ID_FIELD, []))
                
                bad_ref_mask = split_data[POSITIVES_FIELD].apply(has_bad_refs)
                cleaned_split = split_data[~bad_ref_mask].copy()
                
                # Добавление labels для тестового набора
                if split == 'test' and 'labels' not in cleaned_split.columns:
                    cleaned_split['labels'] = [
                        [1 if i == idx else 0 for i in range(len(cleaned_split))] 
                        for idx in range(len(cleaned_split))
                    ]
                
                cleaned_splits[split] = Dataset.from_pandas(cleaned_split.reset_index(drop=True))
            
            # 3. Обновление корпуса (удаление плохих записей)
            if bad_ids:
                corpus_df = corpus_df[~corpus_df[ID_FIELD].isin(bad_ids)]
                corpus = Dataset.from_pandas(corpus_df.reset_index(drop=True))
            
            # 4. Сохранение очищенных данных
            if save_cleaned:
                cleaned_dir = os.path.join(dataset_dir, 'cleaned')
                os.makedirs(cleaned_dir, exist_ok=True)
                
                DatasetDict(cleaned_splits).save_to_disk(os.path.join(cleaned_dir, DATASET_FILE))
                corpus.save_to_disk(os.path.join(cleaned_dir, CORPUS_FILE))
                
                if verbose:
                    print(f"Saved cleaned version to {cleaned_dir}")
            
            bad_ids_report[name] = bad_ids
            
        except Exception as e:
            print(f"Error processing {name}: {str(e)}")
            bad_ids_report[name] = set()
    
    return bad_ids_report
